# Visualizations

Plotly charts from categorized discovery records.
Run `notebook/analysis.ipynb` first so CSVs exist under `output/processed/`.

Paths are anchored at the project root (`simple/`), one level above this notebook.


In [1]:

%pip install -q -r {_requirements()}
%pip install -q "nbformat>=4.2.0"


zsh: parse error near `}'
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# 0. Imports & paths


In [2]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import umap

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots


def _project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "scripts" / "sentiment_analysis.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find project root (expected scripts/sentiment_analysis.py). "
        f"cwd={here}"
    )


ROOT = _project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.categorize_records import EMBEDDING_MODEL

CATEGORY_ORDER = [
    "Workflow & Business Processes",
    "Case Management",
    "Data Management & Visibility",
    "Records & Document Management",
    "System Integration",
    "Reporting & Decision Support",
    "Communication & Collaboration",
    "Scheduling & Resource Management",
    "User Experience & Performance",
    "Training & Documentation",
]

PROCESSED = ROOT / "output" / "processed"
FIG_DIR = ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CATEGORIZED_CHALLENGES_CSV = PROCESSED / "categorized_challenges.csv"
CATEGORIZED_EXPECTATIONS_CSV = PROCESSED / "categorized_expectations.csv"

CHALLENGE_TEXT_COL = "pain_points"
EXPECTATION_TEXT_COL = "expectations"


/Users/lilscott/Code/IPS/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Load categorized records

Reads `categorized_challenges.csv` / `categorized_expectations.csv` written by analysis.
The semantic map cell re-embeds challenges (embeddings are not persisted).


In [3]:
missing = [p for p in (CATEGORIZED_CHALLENGES_CSV, CATEGORIZED_EXPECTATIONS_CSV) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Run notebook/analysis.ipynb through the save step first. Missing:\n  - "
        + "\n  - ".join(str(p) for p in missing)
    )

df = pd.read_csv(CATEGORIZED_CHALLENGES_CSV)
expectations_df = pd.read_csv(CATEGORIZED_EXPECTATIONS_CSV)

print(f"Challenges: {len(df)} | Expectations: {len(expectations_df)}")
print(f"Categories: {df['Category'].nunique()} | Focus groups: {df['focus_group'].nunique()}")


Challenges: 1151 | Expectations: 540
Categories: 10 | Focus groups: 20


## 1b. Reconcile categories → `final_category`

Each record has its own hybrid `Category` (with `Category_Confidence`) and belongs
to a cluster whose dominant category is `Cluster_Label`. **Cluster purity** is the
share of a cluster whose per-record `Category` matches that dominant label.

`final_category` trusts the record's own category when it is confident, but adopts
the cluster's label when the record is low-confidence *and* sits in a pure cluster.

In [ ]:
def add_final_category(frame, purity_threshold=0.6, low_conf=3.0):
    """Blend per-record Category with the cluster's dominant label (Cluster_Label).

    cluster purity = share of a cluster whose Category == Cluster_Label. When a
    record's own category is low-confidence (< low_conf) but it lives in a pure
    cluster (purity >= purity_threshold), adopt the cluster label; else keep Category.
    """
    frame = frame.copy()
    match = frame["Category"] == frame["Cluster_Label"]
    purity = match.groupby(frame["Cluster"]).transform("mean")
    frame["Cluster_Purity"] = np.where(frame["Cluster"] == -1, np.nan, purity)

    frame["final_category"] = frame["Category"]
    take_cluster = (
        (frame["Cluster"] != -1)
        & (frame["Cluster_Purity"] >= purity_threshold)
        & (frame["Category_Confidence"] < low_conf)
    )
    frame.loc[take_cluster, "final_category"] = frame.loc[take_cluster, "Cluster_Label"]
    frame["Category_Reassigned"] = frame["final_category"] != frame["Category"]
    return frame


df = add_final_category(df)
expectations_df = add_final_category(expectations_df)

for label, frame in [("Challenges", df), ("Expectations", expectations_df)]:
    n = int(frame["Category_Reassigned"].sum())
    print(f"{label}: {n}/{len(frame)} records took the cluster label "
          f"({100 * n / len(frame):.1f}%)")

In [ ]:
# Compare cluster purity against category confidence, per cluster (challenges).
purity_summary = (
    df[df["Cluster"] != -1]
    .groupby(["Cluster", "Cluster_Label"])
    .agg(
        records=("Category", "size"),
        purity=("Cluster_Purity", "first"),
        mean_confidence=("Category_Confidence", "mean"),
        reassigned=("Category_Reassigned", "sum"),
    )
    .reset_index()
    .sort_values("purity", ascending=False)
)
display(purity_summary.round(3))

fig_pc = px.scatter(
    df[df["Cluster"] != -1],
    x="Category_Confidence", y="Cluster_Purity",
    color="Cluster_Label",
    hover_data=["focus_group", "Category", "final_category"],
    title="Cluster purity vs category confidence (challenges)",
)
fig_pc.add_hline(y=0.6, line_dash="dash", line_color="#94A3B8", annotation_text="purity 0.6")
fig_pc.add_vline(x=3.0, line_dash="dash", line_color="#94A3B8", annotation_text="confidence 3.0")
fig_pc.update_layout(height=520, width=1000, template="plotly_white")
fig_pc.show()

## 2. Category counts


In [4]:
category_counts = (
    df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_cat = px.bar(
    category_counts, x="Count", y="Category", orientation="h",
    title="Pain point categories", color="Count", color_continuous_scale="Blues",
)
fig_cat.update_layout(showlegend=False, height=500, width=1200, template="plotly_white")
fig_cat.show()

expectation_counts = (
    expectations_df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_exp_cat = px.bar(
    expectation_counts, x="Count", y="Category", orientation="h",
    title="Expectation categories", color="Count", color_continuous_scale="Teal",
)
fig_exp_cat.update_layout(showlegend=False, height=500, width=1200, template="plotly_white")
fig_exp_cat.show()


## 4. Challenges vs expectations by theme


In [5]:
ch_c = df["Category"].value_counts()
ex_c = expectations_df["Category"].value_counts()
cats = sorted(set(ch_c.index) | set(ex_c.index), key=lambda c: -(ch_c.get(c, 0) + ex_c.get(c, 0)))
plot_cmp = pd.DataFrame({
    "Category": cats,
    "Challenges": [int(ch_c.get(c, 0)) for c in cats],
    "Expectations": [int(ex_c.get(c, 0)) for c in cats],
}).melt(id_vars="Category", var_name="Type", value_name="Count")
fig_cmp = px.bar(
    plot_cmp, x="Count", y="Category", color="Type", barmode="group", orientation="h",
    title="Challenges vs expectations by theme",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
)
fig_cmp.update_layout(height=520, template="plotly_white", yaxis=dict(categoryorder="total ascending"))
fig_cmp.show()


## 5. Top focus groups


In [6]:
top_n = 20
top = df.groupby("focus_group").size().sort_values(ascending=False).head(top_n)
exp = expectations_df.groupby("focus_group").size()
plot_top = pd.DataFrame({
    "Focus Group": top.index.tolist(),
    "Challenges": top.values.astype(int),
    "Expectations": [int(exp.get(g, 0)) for g in top.index],
}).melt(id_vars="Focus Group", var_name="Kind", value_name="Count")
fig_top = px.bar(
    plot_top,
    x="Focus Group",
    y="Count",
    color="Kind",
    barmode="group",
    title=f"Top {top_n} focus groups",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
    category_orders={"Focus Group": top.index.tolist()},
)
fig_top.update_layout(
    height=560,
    width=1000,
    template="plotly_white",
    xaxis=dict(tickangle=-40),
    margin=dict(b=140),
)
fig_top.show()


## 6. Focus group × category heatmap


In [ ]:
heatmap_df = pd.crosstab(df["focus_group"], df["final_category"])
focus_group_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[focus_group_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Focus Group", y="Final category", color="Count"),
    color_continuous_scale="YlOrRd", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(height=700, width=1100, xaxis_tickangle=-45, template="plotly_white")
fig_heatmap.show()

In [ ]:
heatmap_df = pd.crosstab(expectations_df["focus_group"], expectations_df["final_category"])
focus_group_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[focus_group_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Focus Group", y="Final category", color="Count"),
    color_continuous_scale="Teal", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(height=700, width=1100, xaxis_tickangle=-45, template="plotly_white")
fig_heatmap.show()

In [ ]:
# Interactive drill-down that works in the Cursor / VS Code notebook renderer.
# (go.FigureWidget.on_click heatmap callbacks only fire in classic Jupyter/JupyterLab,
#  so we render a normal heatmap and drive the record lookup with ipywidgets dropdowns.)
from ipywidgets import interact, Dropdown
from IPython.display import display


def heatmap_explorer(frame, text_col, noun, colorscale, out_html, category_col="final_category"):
    heat = pd.crosstab(frame["focus_group"], frame[category_col])
    heat = heat.reindex(columns=[c for c in CATEGORY_ORDER if c in heat.columns])
    heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index].T

    x_labels = heat.columns.tolist()   # focus groups
    y_labels = heat.index.tolist()     # final categories
    z = heat.values.astype(int)
    text = [[("" if v == 0 else str(v)) for v in row] for row in z]

    fig = go.Figure(go.Heatmap(
        z=z, x=x_labels, y=y_labels, text=text, texttemplate="%{text}",
        textfont=dict(size=11, color="#1a1a1a"),
        colorscale=colorscale, colorbar=dict(title="Count"),
        hovertemplate="<b>%{y}</b><br>%{x}<br>Count: %{z}<extra></extra>",
        xgap=1, ygap=1,
    ))
    fig.update_layout(
        title=f"{noun} volume: focus group × final category",
        template="plotly_white",
        font=dict(family="IBM Plex Sans, Helvetica, Arial, sans-serif", size=13),
        height=max(480, 36 * len(y_labels) + 160),
        margin=dict(l=40, r=30, t=60, b=120),
        xaxis=dict(title="Focus group", tickangle=-40, side="bottom"),
        yaxis=dict(title="Final category", autorange="reversed"),
    )
    fig.write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved {out_html}")
    fig.show()

    detail_cols = [
        c for c in ["focus_group", text_col, "final_category", "Category",
                    "Category_Confidence", "Cluster_Label", "Cluster_Purity"]
        if c in frame.columns
    ]
    ALL = "All"

    def show_records(focus_group, final_category):
        mask = pd.Series(True, index=frame.index)
        if focus_group != ALL:
            mask &= frame["focus_group"] == focus_group
        if final_category != ALL:
            mask &= frame[category_col] == final_category
        subset = frame.loc[mask, detail_cols].drop_duplicates()
        if "Category_Confidence" in subset.columns:
            subset = subset.sort_values("Category_Confidence", ascending=False)
        print(f"{len(subset)} {noun.lower()} records · {focus_group} × {final_category}")
        display(subset if not subset.empty else "No records for this selection.")

    interact(
        show_records,
        focus_group=Dropdown(options=[ALL] + x_labels, value=ALL, description="Focus group:"),
        final_category=Dropdown(options=[ALL] + y_labels, value=ALL, description="Final cat.:"),
    )


# Challenges
heatmap_explorer(
    df, CHALLENGE_TEXT_COL, "Challenge", "YlOrRd",
    FIG_DIR / "heatmap_focus_category.html",
)

In [ ]:
# Expectations (same interactive explorer, expectations dataset)
heatmap_explorer(
    expectations_df, EXPECTATION_TEXT_COL, "Expectation", "Teal",
    FIG_DIR / "heatmap_focus_category_expectations.html",
)